In [2]:
with open('tinyShakeSpeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(f"Length of dataset in characters: {len(text)}")

Length of dataset in characters: 1115394


In [4]:
print("text:", text[:1000])

text: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for reve

In [5]:
# Getting all the unique characters in the text that model can see or generate
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("all the unique characters:", ''.join(chars))
print(f"Vocab size: {vocab_size}")


all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


In [6]:
# Tokenizing the text
# Will be building both encoder and decoder
# Very simple character level tokenizer
stoi = { ch:i for i,ch in enumerate(chars) } # string to integer
itos = { i:ch for i,ch in enumerate(chars) } # integer to string
def encode(s):
    return [stoi[c] for c in s] # encoder: take a string, output a list of integers
def decode(l):
    return ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string
print(encode("hello world"))
print(decode(encode("hello world")))

# We can also use SentencePiece or BPE tokenizers from HuggingFace for the same
# But for this tiny dataset, char level tokenizer is good enough

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


# Understanding Byte Pair Encoding (BPE) -> Tiktoken (OpenAI)

Byte Pair Encoding (BPE) is a subword tokenization algorithm that originated as a data compression technique. In the context of Natural Language Processing (NLP), it's a clever way to represent words by breaking them down into smaller, more manageable units. This method allows models to handle vast vocabularies and words they haven't seen before.

At its core, BPE works by iteratively merging the most frequently occurring pair of adjacent characters or character sequences in a training corpus. This process starts with a vocabulary of individual characters and progressively builds a vocabulary of subword units.

### How BPE Works: A Step-by-Step Guide

Here's a simplified step-by-step breakdown of how the BPE algorithm works:

1.  **Initialization**: The vocabulary is initialized with all the individual characters present in the training text.
2.  **Frequency Counting**: The algorithm counts the occurrences of all adjacent pairs of symbols (characters or sequences) in the text.
3.  **Merging**: The most frequent pair is merged into a single new symbol, and this new symbol is added to the vocabulary.
4.  **Iteration**: The text is updated with the new symbol, and steps 2 and 3 are repeated for a predetermined number of merges or until the desired vocabulary size is reached.

For example, if the pair "e" and "r" appears most frequently, they are merged into "er". In the next iteration, if "er" and " " (space) are the most common pair, they become "er ". This continues until common words or word parts become single tokens.

---

### Why is BPE Used in Tokenization? 🗣️

Traditional tokenization methods that split text by words run into problems with large vocabularies and out-of-vocabulary (OOV) words—words not seen during training. BPE elegantly addresses these issues:

* **Manages Vocabulary Size**: By creating a vocabulary of subword units, BPE can represent a vast number of words with a much smaller, fixed-size vocabulary. This is more memory-efficient for language models.
* **Handles Out-of-Vocabulary Words**: If a model encounters an unknown word, it can often be broken down into known subword units from the BPE vocabulary. For instance, the model might not know "embiggen," but if "em," "bigg," and "en" are in its vocabulary, it can still process the word.
* **Captures Morphological Information**: BPE can learn meaningful subword units like prefixes (e.g., "un-") and suffixes (e.g., "-ing"). This helps the model understand the relationships between words like "run" and "running."

---

### Alternatives to Byte Pair Encoding ↔️

While BPE is widely used, several other subword tokenization algorithms have been developed, each with its own nuances:

* **WordPiece**: Very similar to BPE, WordPiece is used by models like BERT. The key difference is in the merge strategy. Instead of merging the most frequent pair, WordPiece merges the pair that maximizes the likelihood of the training data.
* **SentencePiece**: Developed by Google, SentencePiece treats the input text as a raw stream of characters, including spaces. This makes it language-agnostic and particularly useful for languages that don't use spaces to separate words. It can use either BPE or a unigram language model for tokenization.
* **Unigram Language Model**: Unlike the additive approach of BPE and WordPiece, the unigram model starts with a large vocabulary of possible subwords and iteratively removes the ones that are least likely to occur, based on a probabilistic language model, until the desired vocabulary size is reached.

---

### Drawbacks of Byte Pair Encoding 📉

Despite its advantages, BPE is not without its limitations:

* **Greedy Approach**: The greedy nature of merging the most frequent pair at each step doesn't guarantee the optimal set of subword tokens for the entire corpus. A locally optimal choice might not be globally optimal.
* **Pre-tokenization Dependency**: The effectiveness of BPE can be influenced by the initial pre-tokenization step (e.g., splitting by spaces or punctuation). Different pre-tokenization rules can lead to different final vocabularies.
* **Limited Linguistic Foundation**: The merges in BPE are based purely on frequency, not on any linguistic or semantic principles. This can sometimes result in subword splits that are not linguistically meaningful.
* **Fixed Vocabulary**: Once the vocabulary is trained, it is fixed. This can be a problem for dynamically evolving language or domain-specific jargon that was not present in the initial training data.

In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # first 1000 characters represented as integers


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
# Splitting the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train
train_data = data[:n]
val_data = data[n:]
print(train_data.shape, val_data.shape)
# We use validation set and not test set, this is because we will be iterating a lot on this dataset
# and we will use it  to tune the hyperparameters of the model (test set just gives a final unbiased estimate of the model performance)

torch.Size([1003854]) torch.Size([111540])


In [9]:
# Now we want to plug these sequence of integers into our model for it to learn to predict the next character
# We won't be feeding the entire sequence to the model at once -> computationally very expensive, 
# instead we will break it into chunks of smaller sequences -> block_size (also called context length)
block_size = 8 # how many characters the model can see in the past
train_data[:block_size+1] # we will use first 8 characters to predict the 9th character
# tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]) -> each of these will be used to predict the next character (18 -> 47, 18, 47 -> 56 and so on)

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size] # input
y = train_data[1:block_size+1] # target
for t in range(block_size):
    context = x[:t+1] # the context the model can see
    target = y[t] # the target the model has to predict
    print(f"when input is {context} the target: {target} ({itos[target.item()]})")

# Each integer clubs with the next integer to predict the next integer and at the end of the block size, the transformer starts truncating the context to keep the context length fixed
# For example, when predicting the 9th character, the model can only see characters from 2nd to 8th character (8 character context length)
# This is called causal language modeling (unidirectional)
# The model is not allowed to look into the future characters, only past characters

when input is tensor([18]) the target: 47 (i)
when input is tensor([18, 47]) the target: 56 (r)
when input is tensor([18, 47, 56]) the target: 57 (s)
when input is tensor([18, 47, 56, 57]) the target: 58 (t)
when input is tensor([18, 47, 56, 57, 58]) the target: 1 ( )
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15 (C)
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47 (i)
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58 (t)


## 3D Tensors in GPT-type Chatbots: The Movie Script Analogy

Imagine you're training a movie director chatbot. The **3D tensor** is like a stack of movie scripts it's studying.

| Dimension | Analogy | Significance |
| :--- | :--- | :--- |
| **Batch Size** | The number of scripts you're reading at once. | This is about **computational efficiency**. Instead of processing one script at a time, you can handle multiple scripts (a "batch") simultaneously, making training faster. |
| **Sequence Length** | The number of words in a single script. | This is about **context and order**. It's the maximum number of words the model can look at to understand a sentence. The model learns that "The cat sat on the mat" is a different sequence than "The mat sat on the cat." |
| **Embedding Size** | The meaning of each word. | This is about **semantic meaning**. Each word is represented by a vector of numbers (an embedding) that captures its meaning and relationship to other words. A higher embedding size means a richer, more detailed understanding of each word's semantics. |

The tensor combines these three concepts:

- **Batch Size:** A stack of scripts.
- **Sequence Length:** Each script in the stack is a sequence of words.
- **Embedding Size:** Each word in the sequence has a detailed, numerical representation of its meaning.

Think of it as a 3D grid:
- **X-axis:** The words in a single sequence (Sequence Length).
- **Y-axis:** The features of a single word's meaning (Embedding Size).
- **Z-axis:** The stack of different sequences being processed together (Batch Size).

---

## Concepts Covered in the Code Block below

The code and comments you provided illustrate three core concepts of training large language models: **batching**, **context windows**, and **autoregressive training**.

---

### 1. Batching

**Batching** is a technique to process multiple independent sequences of data in parallel. This is done to significantly increase the efficiency of training. Instead of a single sequence being processed at a time, a **batch** of sequences is processed simultaneously, leveraging the parallel processing power of GPUs.

* The code snippet demonstrates this with `batch_size = 4`. It generates four independent sequences, and the `torch.stack` operation combines them into a single tensor of shape `(4, 8)`.

---

### 2. Context Window (Block Size)

The `block_size` parameter defines the **maximum context length** the model can use to make a prediction.

* A language model doesn't just predict the next character based on the single previous one. It considers a window of preceding characters to understand the context. The `block_size = 8` in the code means the model will use up to 8 characters to predict the 9th. This is crucial for capturing short-range dependencies and improving the quality of predictions.

---

### 3. Autoregressive Training (Inputs and Targets)

The code demonstrates how a model learns to predict the next token in a sequence, a process known as **autoregressive training**.

* The model is trained to predict the next character given all the previous characters in the sequence. This is the core principle of how models like GPT generate text: by predicting one token at a time in a loop.
* The code explicitly sets this up by creating two tensors: `xb` (the inputs) and `yb` (the targets). The `yb` tensor is simply the `xb` tensor shifted by one position. This means for any given input sequence, the model's task is to predict the next character for every character in that sequence. This allows the model to learn efficiently from the entire sequence in a single training step.

In [11]:
# Introducing batch dimension, batches can be defined as independent sequences of characters that the model can learn from in parallel
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences we will process in forward backward pass in parallel?
block_size = 8 # what is the maximum context length for predictions?

def batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data # choose the right dataset
    ix = torch.randint(len(data) - block_size, (batch_size,)) # random starting indices for each of the sequences in the batch
    # torch.randint(len(data) - block_size, (batch_size,)) -> generates a tensor of shape (batch_size,) with random integers between 0 and len(data) - block_size -> so if batch_size = 4, it will generate 4 random starting indices and each of these indices will be the starting point of a sequence of length data - block_size
    x = torch.stack([data[i:i+block_size] for i in ix]) # (batch_size, block_size)
    #torch.stack([data[i:i+block_size] for i in ix]) -> for each of the starting indices in ix, it takes a slice of data from i to i+block_size and stacks them together to form a tensor of shape (batch_size, block_size) 
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # (batch_size, block_size)
    #torch.stack([data[i+1:i+block_size+1] for i in ix]) -> for each of the starting indices in ix, it takes a slice of data from i+1 to i+block_size+1 and stacks them together to form a tensor of shape (batch_size, block_size)

    # x represents the input sequences and y represents the target sequences (next character for each character in the input sequence)
    return x, y
xb, yb = batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

# We get 4 rows and 8 columns, each row is an independent sequence of characters and each column is a character in the sequence
# 8 -> block_size and 4 -> batch_size
# For each of the sequences in the batch, the target is the input sequence shifted by one in simpler words the target is the next character the model has to predict
# For example, in the first row, the input sequence is "When shall I" and the target sequence is "hen shall I " (shifted by one)
# So when the input is "W", the target is "h", when the input is "Wh", the target is "e" and so on
# The model will learn to predict the next character given the previous characters in the sequence
# This is how the model will learn to generate text, by predicting the next character given the previous characters in the sequence

for b in range(batch_size): # iterate over the batch dimension
    for t in range(block_size): # iterate over the block size dimension
        context = xb[b, :t+1] # the context the model can see
        target = yb[b, t] # the target the model has to predict
        print(f"when input is {context} the target: {target} ({itos[target.item()]})")
    print() # add a newline between batches

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is tensor([24]) the target: 43 (e)
when input is tensor([24, 43]) the target: 58 (t)
when input is tensor([24, 43, 58]) the target: 5 (')
when input is tensor([24, 43, 58,  5]) the target: 57 (s)
when input is tensor([24, 43, 58,  5, 57]) the target: 1 ( )
when input is tensor([24, 43, 58,  5, 57,  1]) the target: 46 (h)
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target: 43 (e)
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target: 39 (a)

when input is tensor([44]) the target: 53 (o)
when input is tensor([44, 53]) the target: 56 (r)
when input is tensor([44, 53, 56

In [12]:
# Transformer is going to take in (batch_size, block_size) shaped tensor as input and output (batch_size, block_size, vocab_size) shaped tensor as output
# For each of the characters in the input sequence, the model will output a probability distribution over the entire vocabulary (vocab_size)
# The probability distribution will tell us how likely each character in the vocabulary is to be the next character in the sequence
# For example, if the input sequence is "When shal", the model might output a probability distribution that looks like this:
# [0.1, 0.05, 0.2, 0.15, 0.1, 0.05, 0.1, 0.05, 0.25] 
# where each number represents the probability of each character in the vocabulary being the next character in the sequence 

Building a simple neural network based on the concepts above
In language modelling -> Bigram language model
### **Bigram Language Model** 

A bigram model is a simple type of language model that predicts the next word in a sequence based on the **single preceding word**. The name "bigram" comes from "bi," meaning two, and "-gram," referring to a sequence of characters or words. So, a bigram is a sequence of two consecutive words.

### How It Works

A bigram model works by calculating the probability of a word appearing given the word that came before it. This is based on a large text corpus. The probability is calculated as:

$$P(word_2 | word_1) = \frac{Count(word_1, word_2)}{Count(word_1)}$$

* $P(word_2 | word_1)$ is the probability of `word_2` given `word_1`.
* $Count(word_1, word_2)$ is the number of times the pair `word_1` and `word_2` appears together in the text.
* $Count(word_1)$ is the number of times `word_1` appears.

For example, to predict the word after "I am," the model looks at all instances of "am" in its training data and calculates the probability of the next word being "going," "a," "happy," etc. It would likely find that "going" is the most common word to follow "am" in the corpus.

### Example: "I want to eat"

Consider a corpus with the following sentences:

* "I want to eat pizza"
* "I want to eat sushi"
* "I want to eat a burger"
* "I want to drink soda"

The bigrams and their counts would be:

* ("want", "to"): 3
* ("to", "eat"): 3
* ("eat", "pizza"): 1
* ("eat", "sushi"): 1
* ("eat", "a"): 1
* ("to", "drink"): 1

To determine the probability of the sequence "I want to eat" using a bigram model:

$$P(\text{I want to eat}) \approx P(\text{I}) \times P(\text{want}|\text{I}) \times P(\text{to}|\text{want}) \times P(\text{eat}|\text{to})$$

The calculation for $P(\text{eat}|\text{to})$ would be:

$$P(\text{eat}|\text{to}) = \frac{\text{Count}(\text{to, eat})}{\text{Count}(\text{to})} = \frac{3}{3 + 1} = \frac{3}{4} = 0.75$$

### Applications of Bigram Models

Bigram models provide a strong baseline for many NLP tasks, despite their simplicity:

* **Predictive text and auto-complete:** They predict the next word based on the previous one.
* **Speech recognition:** Bigram probabilities help correct speech-to-text conversion errors by favoring more likely word sequences.
* **Spelling correction:** The models can identify and correct misspelled words that are grammatically incorrect in a specific context.
* **Text generation:** They can generate new sentences by iteratively sampling the next word based on the bigram probabilities.

### Limitations

While simple and fast, bigram models have significant limitations:

1.  **Limited Context:** They only consider the immediate previous word. They can't capture longer-range dependencies, so they would fail to understand context in sentences like "The man who ate the sandwich and then went for a walk was happy." The model would only consider "was" when predicting "happy," ignoring the rest of the sentence.
2.  **Sparsity:** If a pair of words never appeared together in the training data, the model assigns it a probability of zero. This is a problem for unseen word combinations. Smoothing techniques are used to address this, but it remains a fundamental weakness.

Bigram models are a foundational concept in natural language processing but have been largely replaced by more powerful models like transformers, which can process much larger contexts.

# Code and Concept Breakdown

Here's a detailed walkthrough of the `BigramLanguageModel` class and what's happening under the hood.

---

### The `__init__` Method: Setting Up the Lookup Table

**Python**

```python
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
```

`nn.Module`: This is the base class for all neural network modules in PyTorch. Our model inherits from it.

`vocab_size`: This is the number of unique characters in our text. In this case, it's 65.

`self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)`: This is the heart of the Bigram model. Let's break it down:

- `nn.Embedding` creates a simple lookup table. It's essentially a tensor (a multi-dimensional array).
- The first argument, `vocab_size (65)`, defines the number of rows in the table. Each row corresponds to a unique character in our vocabulary.
- The second argument, `vocab_size (65)`, defines the number of columns. Each column represents a score for a possible next character.

So, `nn.Embedding(65, 65)` creates a 65x65 table. When we give it a character's index (e.g., index 24 for the character 'F'), it looks up the 24th row. This row contains 65 numbers. These 65 numbers are the logits—the model's raw predictions for what character comes next.

In this specific Bigram model, the embedding table is **both the lookup table and the prediction layer in one**. It's a shortcut that only works because the model is so simple.

Here’s the conceptual breakdown of why the embedding table is 65x65:

### **Rows vs. Columns: Input vs. Output**
Think of the `nn.Embedding(65, 65)` table's two dimensions as having distinct jobs:

- **The 65 Rows (Number of Embeddings):** This dimension is for the input. Each row corresponds to one of the 65 unique characters in your vocabulary. When the model receives an input character (e.g., the character 'H', which is token index 32), it looks up the 32nd row.

- **The 65 Columns (Embedding Dimension):** This dimension is for the output. The vector stored in each row is the list of logits for the next character. Since there are 65 possible characters that could come next, this vector must be 65 numbers long to provide a score for every possibility.

So, when the model looks up the row for 'H', the 65-dimensional vector it retrieves is directly interpreted as: "Here are the scores for 'A' coming next, 'B' coming next, 'C' coming next, ..., all the way to 'z' coming next."

---

## Understanding the logits Shape: `(4, 8, 65)`

This is a crucial point. The shape `(4, 8, 65)` comes from applying the embedding table to your input batch `xb` which has a shape of `(4, 8)`. Let's decode it dimension by dimension:

- **B (Batch Size = 4):** This is the first dimension. It means we are processing 4 independent chunks of text simultaneously to make the training process more efficient.
- **T (Time / Block Size = 8):** This is the second dimension. Each of the 4 chunks of text is 8 characters long.
- **C (Channels / Vocab Size = 65):** This is the third and final dimension. For each of the 8 characters in each of the 4 chunks, the model looks up its corresponding row in the embedding table. This row, as we discussed, has 65 numbers. These 65 numbers are the scores (logits) for every possible character in the vocabulary that could come next.

So, `logits[0][0]` would be a vector of 65 scores, representing the model's prediction for the character that follows the very first character of the very first sequence in our batch.

---

## The `forward` Method: Making Predictions and Calculating Loss

**Python**

```python
def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx) # (B, T, C)

    if targets is None:
        loss = None
    else:
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)

    return logits, loss
```

This method defines what happens when we pass data through the model.

- `logits = self.token_embedding_table(idx)`: This is the lookup operation we just discussed. It takes our input indices `idx` (shape `(B, T)` or `(4, 8)`) and produces the prediction scores `logits` (shape `(B, T, C)` or `(4, 8, 65)`).

### Calculating Loss:
The loss tells us how bad our model's predictions are. A high loss means bad predictions; a low loss means good predictions.

**Why `F.cross_entropy`?**  
This is the standard loss function for multi-class classification problems. Predicting the next character is essentially a classification problem with 65 classes (our vocabulary). It measures how well the predicted logits align with the correct targets.

**Why reshape with `.view()`?**  
This is a technical requirement of PyTorch's `cross_entropy` function. It expects the predictions in a 2D shape of `(N, C)`, where `N` is the total number of items to classify and `C` is the number of classes.

- Our logits are `(4, 8, 65)`. We have `4 * 8 = 32` characters in total that we are making predictions for. So, we reshape it to `(32, 65)`.
- Our targets are `(4, 8)`. We reshape this to `(32)`—a single list of the 32 correct next characters.

Now, `F.cross_entropy` can compare the 32 sets of 65 predictions against the 32 correct answers and compute a single number representing the average loss.

---

## The `generate` Method: Creating New Text

**Python**

```python
def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        # get the predictions
        logits, loss = self(idx)
        # focus only on the last time step
        logits = logits[:, -1, :] # becomes (B, C)
        # apply softmax to get probabilities
        probs = F.softmax(logits, dim=-1) # (B, C)
        # sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
        # append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx
```

This method uses the trained model to generate new text, token by token.

- **Get Predictions:** It first gets the logits for the entire current sequence `idx`.
- **Focus on the Last Step:** `logits = logits[:, -1, :]` is a key step. We only care about the prediction for what comes after the very last character of our input sequence. This line plucks out just those predictions.
- **Softmax:** The logits are raw scores. `F.softmax` converts these scores into probabilities that all add up to 1. Now, instead of a score like `-2.4`, a character might have a probability of `0.08`.
- **Sample:** `torch.multinomial` samples from this probability distribution. Instead of just picking the character with the highest probability every time (which would be boring and repetitive), it picks a character based on its probability. This introduces some randomness and creativity.
- **Append:** The newly sampled character (`idx_next`) is appended to the input sequence (`idx`), making it one character longer.
- **Repeat:** The loop runs again with the new, longer sequence to predict the next character, and so on, for `max_new_tokens`.

---


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table -> this lookup table was seen in code blocks above
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) 
        # Embedding(num_embeddings, embedding_dim) we use vocab_size for both num_embeddings and embedding_dim because we want to output a probability distribution over the entire vocabulary for each character in the input sequence
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)
        # logits is now (B,T,C) tensor -> C is vocab_size

        # If we have targets, we will compute the loss or else we will just return the logits
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # according to Pytorch documentation, nn.CrossEntropyLoss expects input of shape (N, C) where N is the number of samples and C is the number of classes so we reshape logits from (B, T, C) to (B*T, C)
            targets = targets.view(B*T) # reshape targets from (B, T) to (B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C) -> because we only care about the last time step or last sequence in the context (remember attention block from notes)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) -> torch.multinomial -> idx_next gives you the single, probabilistically chosen next token for each sequence in the batch by sampling from the probability distribution over the vocabulary which means that tokens with higher probabilities are more likely to be chosen but there's still a chance for lower probability tokens to be selected, adding an element of randomness to the generation process
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))
# torch.zeros(1,1) (batch_size, context_length) -> starting with a batch size of 1 and a context length of 1 initialised with <0>, the model will generate 100 new characters based on this initial input
# We take [0] because idk works on the level of batches and we need to unpluck the batch dimension to get the actual sequence of characters (first element of the batch)


"""
Output:
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ"""
# Output is totally gibberish because the model is untrained and also we're using a bigram model which only uses the previous character to predict the next character
# Still our generate function takes entire sequence of past characters and in further code blocks we will see how to use attention mechanism to take into account all the past characters in the context window


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


'\nOutput:\ntensor(4.8786, grad_fn=<NllLossBackward0>)\n\nSKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp\nwnYWmnxKWWev-tDqXErVKLgJ'

In [14]:
# For experimentation let's create an optimizer object
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

# Adam optimizer -> https://www.datacamp.com/tutorial/adam-optimizer-tutorial

""" Adamw is a variant of Adam -> The main difference between Adam and AdamW is their handling of weight decay. Adam incorporates weight decay into the gradient updates, which can interfere with its adaptive learning rate mechanism and affect regularization's efficacy. AdamW, in contrast, decouples weight decay from the gradient updates, applying it directly to the model's parameters after the optimization step, which results in more stable, effective regularization and better model generalization."""

# AdamW -> https://www.datacamp.com/tutorial/adamw-optimizer-in-pytorch



" Adamw is a variant of Adam -> The main difference between Adam and AdamW is their handling of weight decay. Adam incorporates weight decay into the gradient updates, which can interfere with its adaptive learning rate mechanism and affect regularization's efficacy. AdamW, in contrast, decouples weight decay from the gradient updates, applying it directly to the model's parameters after the optimization step, which results in more stable, effective regularization and better model generalization."

In [15]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb, yb = batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 1000 == 0:
        print(steps, loss.item())
    

0 4.692410945892334
1000 3.7637593746185303
2000 3.2342257499694824
3000 2.892245292663574
4000 2.703908681869507
5000 2.515348196029663
6000 2.4889943599700928
7000 2.514069080352783
8000 2.444497585296631
9000 2.3975775241851807


In [16]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))
# Still getting janky output because bigram model is too simple, we need to build a better model with more parameters and attention mechanism to take into account all the past characters in the context window and so that individual characters can influence each other more directly
# But much better than before


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th likl's amen, tofr,
n s Byo tred ceathe, il ivilde w
O ff y
Fivede? ig aiMy, I ivis muofounce herevern outh f athawendesees yof th withS:

FiFLINR:

Wheader y blitow,
Ye m o ditoshyd me, ch rte u hart ararwsa
Wou fe,
INurathoune
IESSARin,
MIOLened sus;
Wh.
S:
NMy BAnind g.
iudshank
An chin is a arokisupxaseru t w ity merwo al LOLo bebte loolld worinero ya l aknge ond thal ttry b's mo ge ck.

gh, inketilllin trk$nutud t ar,
WAnt cithap's Zimponcrdistherdrtes saure ' erpoperrposthel?
Handis of hef t

### **The mathematical trick of Self-Attention**

The best way for the current token to only look at the past tokens is for it to use an average of all the past elements

Karpathy introduces these three methods to explain the **core idea of attention mechanisms** in a simplified, step-by-step manner. The "why" behind this transition is to move from a simple, limited model (the bigram model) to a more powerful and generalizable one that can understand **context**. The ultimate goal is to evolve the model so that each token's prediction isn't based on just one preceding token, but on **all preceding tokens** in the sequence.

### 1. From Bigram to Context
The bigram model is a basic language model where each token's prediction depends *only* on the immediately preceding token. It lacks the ability to understand long-range dependencies or the broader context of a sentence. Karpathy's methods are a way to solve this limitation.

---

### 2. The Intuitive "Bag-of-Words" Mean
The first method with nested `for` loops is the most intuitive. It shows the goal: for any given position `t` in the sequence, the current token should be influenced by the average of all tokens from the start of the sequence up to position `t`. This is a simple form of "contextual understanding." It's computationally inefficient but conceptually clear. This is the **goal state** we want to achieve.

---

### 3. The Efficient Matrix Multiplication
The second method, using a lower triangular matrix, is a more efficient way to achieve the same result as the `for` loops. By creating a lower triangular matrix of ones and then normalizing its rows, you can perform a single matrix multiplication (`weights @ x`) to compute the weighted average for every token in the sequence simultaneously. This is the **computational trick** that makes the process much faster on modern hardware like GPUs, which are highly optimized for matrix operations. The lower triangular shape is crucial because it enforces the "look-back" rule, ensuring that a token at position `t` only "sees" tokens at positions less than or equal to `t`.

---

### 4. The Self-Attention Mechanism
The third method, using `softmax`, is a direct introduction to the **self-attention** mechanism used in Transformers. Instead of a simple mean, it introduces the idea of a **weighted sum** where the weights are learned.
* The `masked_fill` with negative infinity is the clever trick that ensures the model can **only attend to past tokens**. When you apply `softmax` to a row with negative infinities, those values become 0, effectively "masking" future tokens.
* The key difference from the second method is that the `weights` are no longer static. In a real Transformer, these weights are not manually set but are **dynamically calculated** based on the relationship between tokens. This is where the **query, key, and value** vectors come in, which Karpathy will explain later. This allows the model to give more "attention" to certain past tokens that are more relevant to the current token's prediction.

In short, Karpathy is taking you from:
* **Method 1 (Concept):** What we want to do (take a mean of past tokens).
* **Method 2 (Efficiency):** A computationally efficient way to do it using a fixed mean.
* **Method 3 (The Real Deal):** The true foundation of self-attention, where the "mean" is replaced by a dynamic, **learnable weighted sum** that can decide which past tokens are most important for the current one. This is the "why" and the bridge from a simple model to a powerful Transformer.

In [17]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C) #randn generates a tensor of shape (B, T, C) with random values from a normal distribution with mean 0 and variance 1
x.shape

torch.Size([4, 8, 2])

In [ ]:
# We want x[b,t] = mean_(i<=t) x[b, i]
# Version 1: Using for loops
x_bag_of_words = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, c)
        x_bag_of_words[b, t] = torch.mean(xprev, 0) # mean over the time dimension where 0 is the dimension over which we want to take the mean
#x_bag_of_words store the mean of all the previous time steps for each time step in the sequence
x_bag_of_words.shape
x_bag_of_words

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [19]:
x[0] # where x gives two columns representing C=2 features for each of the 8 time steps in the sequence

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [28]:
x_bag_of_words[0] # first op is same but subsequent ops are mean of previous ones
print(x_bag_of_words.shape)

torch.Size([4, 8, 2])


In [21]:
# But this is very inefficient, we can do it with matrix multiplication
# Using weighted aggregation
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a/torch.sum(a, 1, keepdim=True) # normalize the rows -> this works because a is lower triangular matrix and when we divide by the sum of each row, we are effectively taking the mean of all the previous time steps for each time step in the sequence
# torch.tril -> returns the lower triangular part of the matrix
# torch.sum(a, 1, keepdim=True) -> sums the elements of a along dimension 1 (columns) and keeps the dimension of the tensor same by setting keepdim=True
# keepdim=True -> keeps the dimensions of the tensor same after the operation so that we can
b = torch.randint(0, 10, (3,3))
c = a @ b.float() # Convert tensor b to float
print('a=')
print(a)
print('---')
print('b=')
print(b)
print('---')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
---
b=
tensor([[2, 7, 6],
        [4, 6, 5],
        [0, 4, 0]])
---
c=
tensor([[2.0000, 7.0000, 6.0000],
        [3.0000, 6.5000, 5.5000],
        [2.0000, 5.6667, 3.6667]])


In [ ]:
# Version 2: Using weighted aggregation with matrix multiplication

weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True) # normalize the rows -> We're broadcasting the (T, T) weights matrix to (B, T, T) and then doing batch matrix multiplication with (B, T, C) x to get (B, T, C)
print(weights)
x_weighted_average = weights @ x # (T, T) @ (B, T, C) -> (B, T, C) (adds B dimension automatically in (T, T) -> (B, T, T))
print(x_weighted_average.shape)
torch.allclose(x_bag_of_words, x_weighted_average) # checks if two tensors are close to each other within a certain tolerance    

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
torch.Size([4, 8, 2])


False

In [30]:
x_bag_of_words, x_weighted_average

(tensor([[[ 0.1808, -0.0700],
          [-0.0894, -0.4926],
          [ 0.1490, -0.3199],
          [ 0.3504, -0.2238],
          [ 0.3525,  0.0545],
          [ 0.0688, -0.0396],
          [ 0.0927, -0.0682],
          [-0.0341,  0.1332]],
 
         [[ 1.3488, -0.1396],
          [ 0.8173,  0.4127],
          [-0.1342,  0.4395],
          [ 0.2711,  0.4774],
          [ 0.2421,  0.0694],
          [ 0.0084,  0.0020],
          [ 0.0712, -0.1128],
          [ 0.2527,  0.2149]],
 
         [[-0.6631, -0.2513],
          [ 0.1735, -0.0649],
          [ 0.1685,  0.3348],
          [-0.1621,  0.1765],
          [-0.2312, -0.0436],
          [-0.1015, -0.2855],
          [-0.2593, -0.1630],
          [-0.3015, -0.2293]],
 
         [[ 1.6455, -0.8030],
          [ 1.4985, -0.5395],
          [ 0.4954,  0.3420],
          [ 1.0623, -0.1802],
          [ 1.1401, -0.4462],
          [ 1.0870, -0.4071],
          [ 1.0430, -0.1299],
          [ 1.1138, -0.1641]]]),
 tensor([[[ 0.1808, -0.0700]


### How does dividing by weights.sum help give summation of previous rows?

Dividing the `weights` matrix by `weights.sum(1, keepdim=True)` normalizes each row so that it sums to 1. This is crucial because it transforms the matrix multiplication into a **weighted average** of the previous tokens, not just a raw summation.

Let's break down the process:

* **`torch.tril(torch.ones(T, T))`**: This creates a lower triangular matrix. Each row `i` has `i+1` ones, representing the current token and all preceding tokens. For example, for $T=4$:
    $$
    \begin{bmatrix}
    1 & 0 & 0 & 0 \\
    1 & 1 & 0 & 0 \\
    1 & 1 & 1 & 0 \\
    1 & 1 & 1 & 1
    \end{bmatrix}
    $$
* **`weights.sum(1, keepdim=True)`**: This calculates the sum of each row, resulting in a column vector. For the example above, the row sums are `[1, 2, 3, 4]`.
* **`weights / weights.sum(1, keepdim=True)`**: This division operation uses **broadcasting** to divide each element in a row by its corresponding row sum. This results in a matrix where each row sums to 1. The non-zero elements in row `i` are all `1/(i+1)`.
    $$
    \begin{bmatrix}
    1/1 & 0 & 0 & 0 \\
    1/2 & 1/2 & 0 & 0 \\
    1/3 & 1/3 & 1/3 & 0 \\
    1/4 & 1/4 & 1/4 & 1/4
    \end{bmatrix}
    $$

When you compute `x_weighted_average = weights @ x`, the `i`-th row of the normalized `weights` matrix multiplies with the `x` matrix. This effectively takes an average of the first `i+1` token embeddings in `x`, which represents the context for the token at position `i`. This is the core mechanism behind **causal self-attention** in Transformer models.


### What does `weights.sum` give practically?

Practically, `weights.sum(1, keepdim=True)` gives you a **column vector where each element is the number of tokens a given position is allowed to "look at"**.

* `weights` starts as a lower triangular matrix of ones.
* The first row has one '1'. `weights.sum(1)` on this row gives `1`. This means the first token can only look at itself.
* The second row has two '1's. `weights.sum(1)` on this row gives `2`. This means the second token can look at the first two tokens.
* The third row has three '1's. `weights.sum(1)` on this row gives `3`. This means the third token can look at the first three tokens.

This is why dividing the `weights` matrix by this column vector effectively turns the ones in each row into fractions like $1/1, 1/2, 1/3$, etc., ensuring that the weighted sum becomes an **average** of the valid previous tokens. The **single column stretched out concept** is the essence of broadcasting, where this single column of row sums is conceptually repeated across the columns of the `weights` matrix to facilitate the element-wise division.

---

### What is broadcasting here?

**Broadcasting** is a PyTorch mechanism that allows operations on tensors with different shapes by conceptually "stretching" the smaller tensor to match the larger one without creating physical copies. This makes operations more memory and computationally efficient.

In the expression `weights = weights / weights.sum(1, keepdim=True)`, broadcasting is applied to the element-wise division:

* `weights` has a shape of `(T, T)`.
* `weights.sum(1, keepdim=True)` has a shape of `(T, 1)`.

To perform the division, PyTorch compares the shapes from the trailing dimension (right to left):

1.  **Second Dimension**: `weights` has size `T` and `weights.sum` has size `1`. Since one dimension is `1`, PyTorch "stretches" the `(T, 1)` tensor's column to a shape of `(T, T)`. It essentially repeats the column `T` times.
2.  **First Dimension**: Both tensors have a dimension of size `T`. They are already aligned.

This allows the element-wise division to proceed. For each row of `weights`, every element is divided by the single value in the corresponding row of the `weights.sum` tensor.


In [ ]:
# Version 3: Using softmax as a normalization mechanism instead of mean
tril = torch.tril(torch.ones(T, T)) # lower triangular matrix
print(tril)
weights = torch.torch.zeros(T, T)
print(weights)
weights = weights.masked_fill(tril == 0, float('-inf')) # fill the upper triangular part with -inf -> tokens from the past cannot communicate to the future tokens -> basically we are masking the upper triangular part of the matrix with -inf so that when we apply softmax, the values in the upper triangular part become 0.
weights = F.softmax(weights, dim=-1) # apply softmax to each row
print(weights)


tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.200

In [49]:
# Version 4: self-attention mechanism
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C) #randn generates a tensor of shape (B, T, C) with random values from a normal distribution with mean 0 and variance 1

# single head performing self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) -> (B, T, T) -> For every weight this TxT matrix tells us how much each of the tokens in the context should attend to every other token in the context i.e. affinity of each token to every other token -> affinity matrix

tril = torch.tril(torch.ones(T,T))
# wei = torch.zeros((T, T)) We no longer randomly initialise wei to zeros, instead we compute it using the dot product of queries and keys
wei = wei.masked_fill(tril == 0, float('-inf')) # fill the upper triangular part with -inf -> tokens from the past cannot communicate to the future tokens -> basically we are masking the upper triangular part of the matrix with -inf so that when we apply softmax, the values in the upper triangular part become 0.
wei = wei / (head_size ** 0.5) # scale the weights by sqrt(head_size) -> this is done to prevent the dot products from growing too large in magnitude, which can push the softmax function into regions where it has extremely small gradients -> this is called scaled dot-product attention
wei = F.softmax(wei, dim=-1) # (B, T, T) -> apply softmax to each row
v = value(x) # (B, T, 16)
out = wei @ v # (B, T, T) @ (B, T,

print(out.shape)
print(wei)



torch.Size([4, 8, 16])
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
         [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
         [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
         [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4016, 0.5984, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3365, 0.2271, 0.4364, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3019, 0.2060, 0.2899, 0.2022, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1058, 0.1700, 0.1530, 0.3451, 0.2261, 0.0000, 0.0000, 0.0000],


### **Notes:**
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.

- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens. Attention is different from Convolution operation. Convolution actually works with spatial information whereas Attention works with nodes that are randomly placed somewhere in the space and too get some information about their position we add positional encoding to the embedding.

- Each example across batch dimension is of course processed completely independently and never "talk" to each other. So basically rather than there being 32 nodes there are 4 batches of 8 nodes each wherein each batch only those 8 nodes communicate with each other.

- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.

- "self-attention" just means that the keys and values are produced from the same source as queries i.e from `x`. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module). Self-attention -> nodes talking to each other and sharing information, Cross-attention -> pulling information from external nodes so our current nodes can use them

- "Scaled" attention additional divides `wei` by $1 / \sqrt{head\_size}$. This makes it so when input Q,K are unit variance, `wei` will be unit variance too and Softmax will stay diffuse and not saturate too much. We don't want the Softmax to peak and get sharp towards the highest value, we want the `wei` to be diffused so as to control scaling during initialisation